# Duckie-Detection — Analyse, Probleme & Fix-Ideen
**DuckieRace · Session 2026-06-16**

Dieses Notebook dokumentiert die Untersuchung der Entenerkennung (`detect_duckies_node`,
YOLO/ONNX) inkl. der gefundenen Freeze-Ursache, der Messdaten als Beleg, der Fix-Ansätze
(mit Code-Snippets) und eines neu gefundenen Bugs.

> Hinweis: Code-Zellen sind teils **illustrativ** (Snippets/Vorschläge), nicht alle direkt
> lauffähig ohne laufendes ROS. Die Mess-/Plot-Zelle läuft eigenständig (nur `matplotlib`).

## 1. Kontext & Pipeline

`detect_duckies_node`:
1. Kamerabild → resize **640×480** (`cbFindDuckies`), in `_pending_image` ablegen.
2. Separater **Inferenz-Thread** (`_inference_loop`): nimmt das neueste Bild, ONNX-Inferenz
   (`duckie_v19.onnx`), Boxen mit `conf >= conf_threshold` übernehmen.
3. Nächste/unterste Ente wählen; wenn `distance_min <= y2 <= distance_max`:
   Ente per **YOLO-Bounding-Box aus dem HSV-Bild maskieren**, Spur (gelb/weiß) bestimmen,
   Ausweich-Lücke + `duckie_error` berechnen, `duckie_control_active=True`.
4. Verbraucher: `control_lane` (Ausweichen, 10 Hz Regelschleife), `switch_control` (Obstacle-Modus).

Modell-IO (verifiziert): Input `[1,3,480,640]` float RGB /255 · Output `[1,300,6]` = x1,y1,x2,y2,conf,class.

## 2. Problem A — „Enten nicht / zu spät erkannt": was AUSGESCHLOSSEN wurde

| Hypothese | Status |
|---|---|
| **Distanz-Gate** `distance_min=150`, `distance_max=460` (an bbox-Unterkante y2, 0..480) ignoriert weit/zu nah | Laut Felix **nicht** die Ursache (Enten gut im Band sichtbar) |
| **`conf_threshold=0.3`** filtert <0.3 | conf ist **bimodal** (~0 oder 0.7–0.93); Verpasser sind meist ~0 → Schwelle hilft kaum |
| **Vorverarbeitung falsch?** | **Korrekt verifiziert** (Input-Form, RGB, /255 passen) → kein Format-Bug |

→ Das eigentliche Problem war nicht die Erkennung selbst, sondern **Freezes** (siehe 3).

## 3. Problem B — die „10-Sekunden-Freezes" → URSACHE GEFUNDEN

**Symptom:** Bot + Ente statisch, Ente ~10 s nicht erkannt, dann plötzlich wieder.

**Beobachtungen / Ausschluss:**
- ROS-Logs: **Apriltag UND Duckie froren gleichzeitig** (~10–14 s Lücken in beiden Logs).
- **Kamera/Netz ausgeschlossen:** Live-Monitor zeigte Kamera durchgehend ~30 fps — auch während
  der Freezes. (ROS_IP war korrekt: Nodes auf 192.168.90.224, tick auf .240.)
- **CPU ausgeschlossen:** 16 Kerne, 62 GB RAM, 0 Swap, kein Container-Limit. Im Freeze **fiel**
  die CPU auf ~28 % statt Sättigung → Nodes **blockierten/warteten**, rechneten nicht.

**Beweis durch Experiment:** `debug_view` gekillt → 100 s lang **kein** Freeze.
→ Ursache: **`cv2.imshow` über X11** blockiert die Node-Loops (zickiges Display), und drückt per
ROS-Backpressure auf die Detektoren. ABER: `detect_duckies` (Z.390) und `detect_lane` (Z.219)
haben **eigene** `cv2.imshow` → zweite Quelle (Freeze trat ohne debug_view erneut in der „Duckie View" auf).

### 3.1 Die Messdaten (Beleg)

Live-Monitor (`/tmp/mon.py`): pro Sekunde Kamera-FPS, Duckie-Hz, Apriltag-Hz, CPU%.
Unten die zwei Läufe um die Freeze-Fenster. **Run 1 = mit `debug_view`** (Freezes bei t≈41–44 und
t≈64–71: `duck=0, apr=0`, aber `cam` bleibt ~30 und CPU **fällt**). **Run 2 = ohne `debug_view`**
(kein Freeze; bei t≈46–65 nur hohe CPU, aber `apr/duck` nie 0).

In [ ]:
import matplotlib.pyplot as plt

# (t, cam_fps, duck_hz, apr_hz, cpu%) -- Auszug um die Freeze-Fenster
# Run 1: MIT debug_view  -> zwei Freezes
run1 = [
 (38,34,4,26,70.7),(39,21,5,27,74.9),(40,14,3,5,53.8),
 (41,16,0,0,25.5),(42,28,0,0,28.0),(43,19,0,0,27.1),(44,36,0,0,29.9),  # FREEZE 1
 (45,26,3,23,48.1),(46,27,5,24,75.4),
 (60,38,3,40,78.7),(61,30,5,26,84.2),(62,28,1,23,55.5),(63,32,0,33,42.7),
 (64,30,0,0,29.5),(65,31,0,0,28.2),(66,29,0,0,30.4),(67,29,0,0,29.0),
 (68,31,0,0,27.3),(69,31,0,0,28.1),(70,30,0,0,31.9),(71,30,0,0,29.8),  # FREEZE 2
 (72,27,4,26,59.3),
]
# Run 2: OHNE debug_view -> kein Freeze (t46-65 nur hohe CPU)
run2 = [
 (44,31,6,30,67.0),(45,30,6,30,70.3),(46,29,4,29,86.9),(47,31,4,30,89.6),
 (48,30,3,28,92.3),(49,30,4,29,96.2),(50,30,5,16,89.4),(51,30,4,17,88.1),
 (52,30,3,38,93.1),(53,30,4,30,94.6),(60,30,2,31,92.9),(64,30,5,28,90.6),
 (65,30,3,27,92.4),(66,30,6,31,70.1),(67,30,6,30,67.5),(68,30,6,29,69.4),
]

def plot(run, title):
    t=[r[0] for r in run]; cam=[r[1] for r in run]; duck=[r[2] for r in run]
    apr=[r[3] for r in run]; cpu=[r[4] for r in run]
    fig,ax=plt.subplots(figsize=(9,3.5))
    ax.plot(t,cam,'o-',label='cam_fps'); ax.plot(t,apr,'s-',label='apr_hz')
    ax.plot(t,duck,'^-',label='duck_hz'); ax.plot(t,cpu,'--',label='cpu%',alpha=0.6)
    ax.set_xlabel('t [s]'); ax.set_title(title); ax.legend(loc='upper right'); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

plot(run1,'Run 1: MIT debug_view -> apr/duck brechen auf 0 ein, cam bleibt ~30, CPU faellt')
plot(run2,'Run 2: OHNE debug_view -> kein Einbruch (t46-65 nur hohe CPU)')


**Lesart:** Im Freeze ist `cam_fps ~30` (Kamera ok) und `cpu` **niedrig** (kein Rechen-Engpass),
aber `apr_hz=duck_hz=0` → beide schweren Nodes **warten**. Das ist die Signatur eines
**Block**, nicht von Netz oder CPU.

## 4. Fix-Ansätze für die Freezes (noch nicht angewendet)

Offene Frage von Felix: *„ist das die einzige Lösung?"* — Nein, vier Wege:

- **(A) Headless-Schalter `DUCKIE_GUI`** (Default aus): alle `cv2.imshow`/`waitKey` nur bei gesetztem
  Flag. Debug-Bilder gibt's eh als ROS-Topics → `rqt_image_view`. Behebt auch die X11 `-6`-Crashes.
- **(B) `imshow` aus dem Verarbeitungs-/Inferenz-Loop in einen eigenen Thread** mit eigener Rate.
  Die Anzeige darf hängen, ohne die Erkennung zu blockieren. *(Favorit)*
- **(C) Debug-Publishes entkoppeln** (non-blocking / drop), damit ein langsamer Viewer die
  Detektoren nicht per Backpressure ausbremst.
- **(D) GUI nur starten, wenn ein schnelles lokales Display vorhanden ist.**

Betroffene Stellen: `detect_duckies_node.py:390`, `detect_lane_node.py:219`, `debug_view_node.py` (imshow).

In [ ]:
# (A) Headless-Schalter -- minimaler Eingriff an jeder imshow-Stelle
import os
# statt:
#     cv2.imshow('Duckie View', img); cv2.waitKey(1)
# ->
if os.environ.get('DUCKIE_GUI') == '1':
    cv2.imshow('Duckie View', img)
    cv2.waitKey(1)
# Default (Variable nicht gesetzt) = headless -> kein X11 -> kein Block, kein -6-Crash.
# Anzeigen via:  rqt_image_view /tick/debug/duckie_view


In [ ]:
# (B) Anzeige in eigenem Thread, entkoppelt vom Inferenz-Loop (Skizze)
import threading, cv2
_latest = {'img': None}
def _viewer():
    while True:
        img = _latest['img']
        if img is not None and os.environ.get('DUCKIE_GUI') == '1':
            cv2.imshow('Duckie View', img); cv2.waitKey(30)
        else:
            cv2.waitKey(30)
# einmal starten:  threading.Thread(target=_viewer, daemon=True).start()
# im Inferenz-Loop NUR noch:  _latest['img'] = img   (nicht-blockierend)


## 5. Modellverhalten & Robustheit

`max_conf` ist **bimodal**: entweder ~0.0 (kein/Verpasser) oder 0.7–0.93 (sichere Erkennung) —
kaum Mittelwerte. Ein verpasster, gut sichtbarer Ente-Frame = Modell gibt ~0 zurück
(Reichweiten-/Robustheitsgrenze: klein/weit/Winkel/Unschärfe). Inferenz ~75 ms (~5–14 Hz).

Milderungen:
- **Hysterese**: Erkennung über N Frames halten → einzelne Konfidenz-Einbrüche kippen sie nicht.
- `conf_threshold` leicht senken (~0.25) — fängt nur die seltenen 0.2er, Fehlalarm-Risiko beachten.
- Besseres/neu trainiertes Modell — der eigentliche Hebel für mehr Reichweite.

In [ ]:
# (5) Detektions-Hysterese (Skizze): Ente gilt als "da" fuer ein paar Frames nach letzter Erkennung
class DuckieHysteresis:
    def __init__(self, hold_frames=5):
        self.hold = hold_frames
        self._left = 0
        self.last_boxes = []
    def update(self, boxes):
        if boxes:
            self.last_boxes = boxes
            self._left = self.hold
        elif self._left > 0:
            self._left -= 1            # halte letzte Boxen
        else:
            self.last_boxes = []
        return self.last_boxes          # -> diese fuer Maskierung/Reaktion nutzen


## 6. NEUER BUG (gefunden 2026-06-16) — Maskierung bricht bei fehlender Bounding-Box

Die Ente wird per YOLO-BB aus dem HSV-Bild maskiert (`hsv[duckie_mask==255]=0`), damit sie die
**Spurerkennung** nicht stört. **Aber:** hat ein Frame **keine** YOLO-BB (Erkennung kippt für diesen
Frame weg — siehe bimodale conf), wird die Ente **nicht** maskiert → ihre **gelbe Farbe** wird als
**gelbe Linie** erkannt → verfälscht Spur-/Ausweich-Logik.

Direkt gekoppelt an das Flacker-Problem (Punkt 5): genau in den Frames, in denen YOLO die Ente verliert.

**Fix-Ideen:**
- Letzte bekannte BB über Frames **halten / leicht vergrößern** für die Maske (Mask-Hysterese,
  siehe Snippet 5), bis YOLO wieder greift.
- Spur-/Ausweichwerte **nur bei stabiler** Entenerkennung übernehmen (sonst letzten Wert halten).
- Ente zusätzlich per **Farbe+Region** ausmaskieren (Fallback, wenn keine BB).

In [ ]:
# (6) Mask-Hysterese gegen "Ente wird zur gelben Linie", wenn BB mal fehlt (Skizze)
# self._duckie_hyst = DuckieHysteresis(hold_frames=5)  # in __init__
boxes = self._duckie_hyst.update(self.final_duckies)   # statt direkt self.final_duckies
duckie_mask = np.zeros(cv_image.shape[:2], dtype=np.uint8)
for x1, y1, x2, y2, conf in boxes:
    # optional leicht vergroessern (Rand), um Unsicherheit abzudecken:
    pad = 6
    duckie_mask[max(0,int(y1)-pad):int(y2)+pad, max(0,int(x1)-pad):int(x2)+pad] = 255
hsv[duckie_mask == 255] = 0


## 7. Ausweich-Logik (Ente umfahren) — ist korrekt

`fnGetLaneDuckieError` nimmt die **größere** Lücke (sichtbare Linie ↔ Enten-Kante) und lenkt zur
Lückenmitte. **Defaults für unsichtbare Linien existieren** (`white_alternative=0.95·B`,
`yellow_alternative=0.05·B` = Bildränder) → unsichtbare Linie gilt als „weit außen" → ihre Seite
wird die größere Lücke → Bot weicht dorthin aus. Entspricht Felix' Intuition („unsichtbare Linie =
mehr Platz") und ist bereits so umgesetzt.

## 8. CPU / Drossel (depriorisiert — war NICHT die Freeze-Ursache)

ONNX nutzt per Default **alle** Kerne und läuft **back-to-back** (`min_inference_interval=0.5` ist
definiert, aber **unbenutzt**). Bei 16 Kernen unkritisch für die Freezes, aber sauberer wäre:
ONNX-Threads begrenzen, Inferenz auf ~Reglerrate drosseln, `apriltag nthreads` von 4 zurück auf 2.

In [ ]:
# (8) ONNX schlanker konfigurieren (Skizze)
import onnxruntime as ort
so = ort.SessionOptions()
so.intra_op_num_threads = 2          # nicht alle Kerne belegen
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
session = ort.InferenceSession('models/duckie_v19.onnx', sess_options=so)

# Drossel im _inference_loop (min_inference_interval endlich nutzen):
# if now - self._last_inference_time < self.min_inference_interval: time.sleep(...); continue


## 9. Diagnose-Tool: `/tmp/mon.py`

rospy-Monitor, der pro Sekunde Kamera-FPS + Duckie/Apriltag-Hz + CPU% ausgibt — trennt
**Netz vs. CPU vs. Block** sauber. Vor dem Start ROS-Umgebung setzen
(`ROS_MASTER_URI`, `ROS_IP`, `source devel`).

In [ ]:
#!/usr/bin/env python3
# /tmp/mon.py  -- Vision-Pipeline-Monitor
import rospy, time, os
from sensor_msgs.msg import CompressedImage
from std_msgs.msg import Bool, Int32

cnt = {'cam':0, 'duck':0, 'apr':0}
def mk(k):
    def cb(m): cnt[k]+=1
    return cb
def cpu_idle_total():
    with open('/proc/stat') as f:
        v = list(map(int, f.readline().split()[1:]))
    return v[3]+v[4], sum(v)

rospy.init_node('mon', anonymous=True, disable_signals=True)
veh = os.environ.get('VEHICLE_NAME') or 'tick'
rospy.Subscriber(f'/{veh}/camera_node/image/compressed', CompressedImage, mk('cam'), queue_size=100)
rospy.Subscriber(f'/{veh}/detect/duckies', Bool, mk('duck'), queue_size=100)
rospy.Subscriber(f'/{veh}/detect/apriltag', Int32, mk('apr'), queue_size=100)

print("t_s  cam_fps  duck_hz  apr_hz  cpu%", flush=True)
pi, pt = cpu_idle_total(); t0 = time.time()
while time.time()-t0 < 100 and not rospy.is_shutdown():
    time.sleep(1.0)
    c, d, a = cnt['cam'], cnt['duck'], cnt['apr']; cnt['cam']=cnt['duck']=cnt['apr']=0
    i, t = cpu_idle_total(); cpu = 100.0*(1-(i-pi)/max(1,(t-pt))); pi, pt = i, t
    print(f"{time.time()-t0:5.1f}  {c:5d}  {d:5d}  {a:5d}  {cpu:5.1f}", flush=True)


## 10. Offene Punkte / nächste Schritte

- [ ] Freeze-Fix wählen & umsetzen: **(B)** imshow in eigenen Thread *oder* **(A)** `DUCKIE_GUI`-Flag.
      Betrifft `detect_duckies_node`, `detect_lane_node`, `debug_view_node`.
- [ ] **Masking-Bug (6)** fixen: Mask-Hysterese, damit fehlende BB nicht zu „gelber Linie" führt.
- [ ] Optional: Detektions-Hysterese (5) gegen Flackern.
- [ ] Optional CPU-Hygiene (8): ONNX-Threads/Drossel, `apriltag nthreads` 4→2.
- [ ] Branch: Align liegt auf `feature/intersection-with-align`; Duckie-/Apriltag-Änderungen noch uncommittet.

*Details auch in der Projekt-Memory: `memory/duckie_detection.md`.*